# Lab 4 - Spark Structured Streaming on the Wikimedia event stream

The producer (`wikistream_producer.py`, run on the host) reads the Wikimedia `recentchange` stream and
sends every event, as JSON, to the Kafka topic `wikistreams`. This notebook is the consumer: Spark
Structured Streaming reads the topic and computes aggregations over event-time windows.

Plan:

1. Demo from the course: number of edits by bots / humans per 1 hour window.
2. Exploration 1, **tumbling windows**: activity per 1 minute window (edits, bytes added / removed).
3. Exploration 2, **overlapping (sliding) windows**: 10 minute windows every 2 minutes, bots vs humans.
4. Exploration 3, **overlapping windows**: most active users over the last 5 minutes, updated every minute.
5. Exploration 4, **filter without aggregation**: big edits, written as soon as they arrive.

The `console` sink writes into the logs of the container (`docker logs -f pyspark_notebook`). The
explorations also write to a `memory` sink, so the results can be read from the notebook with Spark SQL.

In [ ]:
# Imports cell
import os
import time

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_json, window, count, sum as sum_, avg, max as max_, abs as abs_,
    when, coalesce, lit, timestamp_seconds, approx_count_distinct
)
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, IntegerType, BooleanType
)

## Spark session

The Kafka connector is not shipped with Spark, it is downloaded with `spark.jars.packages`. Its version
has to match the Spark version of the image, so it is built from `pyspark.__version__` instead of being
written by hand (the image we pulled runs a more recent Spark than the 4.1.0 of the demo).

In [ ]:
# Kafka address inside the docker network (see compose.yaml). Can be overridden to run outside docker.
KAFKA_BOOTSTRAP = os.environ.get("KAFKA_BOOTSTRAP", "kafka:19092")
TOPIC = "wikistreams"

spark = SparkSession.builder \
    .appName("WikistreamsStructuredStreaming") \
    .config("spark.jars.packages",
            f"org.apache.spark:spark-sql-kafka-0-10_2.13:{pyspark.__version__}") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

# small data, 4 shuffle partitions instead of 200 make every micro-batch much faster
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

In [ ]:
# Define the initial raw streaming dataframe
raw_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP) \
    .option("subscribe", TOPIC) \
    .option("startingOffsets", "earliest") \
    .load()

raw_df.printSchema()

Kafka gives a fixed schema (`key`, `value`, `topic`, `partition`, `offset`, `timestamp`...). The event
itself is in `value`, as bytes. It is cast to a string and parsed with `from_json` and an explicit
schema (a streaming source can not infer it).

Compared to the demo, the schema also keeps `server_name` and `namespace` (0 = article, 1 = talk page,
2 = user page...), used by the explorations.

In [ ]:
wikimedia_schema = StructType([
    StructField("title", StringType(), True),
    StructField("user", StringType(), True),
    StructField("timestamp", LongType(), True),     # event time, unix epoch in seconds
    StructField("type", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("server_name", StringType(), True),
    StructField("namespace", IntegerType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("comment", StringType(), True),
    StructField("length", StructType([
        StructField("old", IntegerType(), True),
        StructField("new", IntegerType(), True)
    ]), True)
])

parsed_df = raw_df.select(
    from_json(col("value").cast("string"), wikimedia_schema).alias("data")
).select("data.*")

# event_time: real timestamp type (from_unixtime of the demo returns a string)
# size_delta: bytes added (> 0) or removed (< 0) by the edit, a new page has no 'old' size
edits_df = parsed_df \
    .withColumn("event_time", timestamp_seconds(col("timestamp"))) \
    .withColumn("size_delta", coalesce(col("length.new"), lit(0)) - coalesce(col("length.old"), lit(0)))

edits_df.printSchema()

## 1. Demo: bot edits per 1 hour window

Code of the demo, with `event_time` as a timestamp. `complete` mode: the whole result table is printed
at every trigger.

In [ ]:
# A simple aggregation of raw data by number of bot edits
windowed_counts_df = edits_df \
    .groupBy(
        window(col("event_time"), "1 hour"),
        col("bot")
    ) \
    .agg(count("*").alias("edit_count")) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count")
    )

# Use console output (goes to Docker logs)
# trigger added: without it the full table is printed after every micro-batch
demo_query = windowed_counts_df \
    .writeStream \
    .queryName("demo_bot_per_hour") \
    .format("console") \
    .outputMode("complete") \
    .trigger(processingTime="30 seconds") \
    .option("truncate", False) \
    .start()

## 2. Exploration 1: tumbling windows of 1 minute

Tumbling window = `window(event_time, "1 minute")` with no slide: the windows do not overlap and each
edit belongs to exactly one window.

For each minute: number of edits, share of bot edits, bytes added, bytes removed, average size of an
edit and the biggest change.

- **Watermark** of 2 minutes: an event arriving more than 2 minutes after the latest event time seen is
  dropped, and Spark can forget the state of the old windows (otherwise the state grows forever).
- Console sink in **update** mode: only the windows that changed in the micro-batch are printed.
- Trigger every 30 seconds instead of "as soon as possible", to get fewer and bigger micro-batches.

In [ ]:
tumbling_df = edits_df \
    .withWatermark("event_time", "2 minutes") \
    .groupBy(window(col("event_time"), "1 minute")) \
    .agg(
        count("*").alias("edits"),
        sum_(when(col("bot"), 1).otherwise(0)).alias("bot_edits"),
        sum_(when(col("size_delta") > 0, col("size_delta")).otherwise(0)).alias("bytes_added"),
        sum_(when(col("size_delta") < 0, -col("size_delta")).otherwise(0)).alias("bytes_removed"),
        avg(abs_(col("size_delta"))).alias("avg_abs_delta"),
        max_(abs_(col("size_delta"))).alias("max_abs_delta")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "edits", "bot_edits", "bytes_added", "bytes_removed",
        "avg_abs_delta", "max_abs_delta"
    )

tumbling_console = tumbling_df.writeStream \
    .queryName("tumbling_console") \
    .format("console") \
    .outputMode("update") \
    .trigger(processingTime="30 seconds") \
    .option("truncate", False) \
    .start()

# same result in a memory table, to read it from the notebook
tumbling_memory = tumbling_df.writeStream \
    .queryName("tumbling_1min") \
    .format("memory") \
    .outputMode("complete") \
    .trigger(processingTime="30 seconds") \
    .start()

## 3. Exploration 2: overlapping windows, 10 minutes every 2 minutes

Sliding window = `window(event_time, "10 minutes", "2 minutes")`: a new window starts every 2 minutes and
lasts 10 minutes, so each edit is counted in 5 windows. The curve is smoother than with tumbling windows,
it works like a moving average over the last 10 minutes.

Bots vs humans: number of edits, number of distinct users and average size of the edits.
`countDistinct` is not supported on a streaming DataFrame, `approx_count_distinct` (HyperLogLog) is.

In [ ]:
sliding_df = edits_df \
    .withWatermark("event_time", "10 minutes") \
    .groupBy(
        window(col("event_time"), "10 minutes", "2 minutes"),
        col("bot")
    ) \
    .agg(
        count("*").alias("edits"),
        approx_count_distinct("user").alias("users"),
        avg(abs_(col("size_delta"))).alias("avg_abs_delta")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "bot", "edits", "users", "avg_abs_delta"
    )

sliding_console = sliding_df.writeStream \
    .queryName("sliding_console") \
    .format("console") \
    .outputMode("update") \
    .trigger(processingTime="30 seconds") \
    .option("truncate", False) \
    .start()

sliding_memory = sliding_df.writeStream \
    .queryName("sliding_10min") \
    .format("memory") \
    .outputMode("complete") \
    .trigger(processingTime="30 seconds") \
    .start()

## 4. Exploration 3: most active users, 5 minute windows every minute

Overlapping windows again, grouped by user. `limit` / `orderBy` on a streaming aggregation are not allowed
in `update` mode, so the counts go to a memory table (complete mode) and the top 10 is done in SQL on it.

In [ ]:
users_df = edits_df \
    .withWatermark("event_time", "5 minutes") \
    .groupBy(
        window(col("event_time"), "5 minutes", "1 minute"),
        col("user"),
        col("bot")
    ) \
    .agg(
        count("*").alias("edits"),
        sum_(col("size_delta")).alias("net_bytes")
    )

users_memory = users_df.writeStream \
    .queryName("users_5min") \
    .format("memory") \
    .outputMode("complete") \
    .trigger(processingTime="30 seconds") \
    .start()

## 5. Exploration 4: big edits (filter, no aggregation)

`select` + `where` work like on a static DataFrame. There is no aggregation, so the query runs in
**append** mode: each big edit (more than 2 000 bytes added or removed) is printed once, as soon as its
micro-batch is processed. Useful to spot a page blanked by a vandal or a big article import.

In [ ]:
big_edits_df = edits_df \
    .where(abs_(col("size_delta")) >= 2000) \
    .select("event_time", "server_name", "title", "user", "bot", "minor", "size_delta", "comment")

big_edits_console = big_edits_df.writeStream \
    .queryName("big_edits_console") \
    .format("console") \
    .outputMode("append") \
    .option("truncate", 60) \
    .start()

big_edits_memory = big_edits_df.writeStream \
    .queryName("big_edits") \
    .format("memory") \
    .outputMode("append") \
    .start()

## Results

The queries run in the background. Let the producer run a few minutes, then read the memory tables
(the cells can be run again to see the tables grow).

In [ ]:
# wait for a few micro-batches
time.sleep(90)

for q in spark.streams.active:
    print(f"{q.name:20s} active={q.isActive}  last batch: {q.lastProgress['numInputRows'] if q.lastProgress else '-'} rows")

In [ ]:
# Exploration 1: activity per minute (tumbling)
spark.sql("""
    SELECT window_start, window_end, edits, bot_edits,
           ROUND(100 * bot_edits / edits, 1) AS bot_pct,
           bytes_added, bytes_removed, ROUND(avg_abs_delta) AS avg_abs_delta, max_abs_delta
    FROM tumbling_1min
    ORDER BY window_start
""").show(30, truncate=False)

In [ ]:
# Exploration 2: bots vs humans over 10 minute sliding windows
spark.sql("""
    SELECT window_start, window_end, bot, edits, users, ROUND(avg_abs_delta) AS avg_abs_delta
    FROM sliding_10min
    ORDER BY window_start, bot
""").show(30, truncate=False)

In [ ]:
# Exploration 3: top 10 users in the most recent complete 5 minute window
spark.sql("""
    WITH last_window AS (
        SELECT MAX(window.start) AS start FROM users_5min
        WHERE window.end <= (SELECT MAX(window.end) FROM users_5min) - INTERVAL 1 MINUTE
    )
    SELECT window.start AS window_start, window.end AS window_end, user, bot, edits, net_bytes
    FROM users_5min
    WHERE window.start = (SELECT start FROM last_window)
    ORDER BY edits DESC, net_bytes DESC
    LIMIT 10
""").show(truncate=False)

In [ ]:
# Exploration 4: biggest edits received so far
spark.sql("""
    SELECT event_time, server_name, title, user, bot, size_delta
    FROM big_edits
    ORDER BY ABS(size_delta) DESC
    LIMIT 10
""").show(truncate=40)

## Stop the queries

A streaming query never ends on its own. Stop them before closing the notebook (or before running
the cells again, otherwise two queries with the same `queryName` can not run at the same time).

In [ ]:
for q in spark.streams.active:
    print("stopping", q.name)
    q.stop()

## Notes

- **Tumbling vs sliding**: with a 1 minute tumbling window each edit is counted once and we see the peaks
  minute by minute. With a 10 minute / 2 minute sliding window each edit is counted 5 times; the values
  are bigger and smoother, closer to a "last 10 minutes" indicator.
- **Output modes**: `complete` for the small tables read in the notebook, `update` for the console (only
  the changed windows), `append` for the filter query (a row never changes once written). `append` on an
  aggregation would only print a window once the watermark has passed its end.
- **Watermark**: without it the state of every window is kept forever. With it, late events (more than
  the watermark behind the max event time) are dropped and old windows are removed from the state.
- **Not supported in streaming**: `distinct` / `countDistinct` (replaced by `approx_count_distinct`),
  `limit` and `orderBy` outside complete mode (done in SQL on the memory table instead).
- **Producer filters**: the wiki, the change types, the namespace, the pages and bots / humans are chosen
  on the command line of `wikistream_producer.py` (see the README). The notebook does not change: it reads
  whatever the producer sends to the topic.